# South Florida Environmental Monitoring + GeoAI
This notebook is a starter workflow for a portfolio project integrating satellite remote sensing, environmental monitoring, time-series analysis, flood mapping, land-cover classification, and GIS automation.

In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

## 1. Define study area
Replace the geometry below with a county, watershed, or uploaded boundary.

In [ ]:
aoi = ee.Geometry.Rectangle([-80.90, 25.10, -80.00, 26.20])
Map = geemap.Map(center=[25.65, -80.45], zoom=8)
Map.addLayer(aoi, {}, 'AOI')
Map

## 2. Sentinel-2 composite and spectral indices

In [ ]:
def mask_s2_clouds(img):
    scl = img.select('SCL')
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return img.updateMask(mask).divide(10000)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2025-01-01', '2025-12-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
      .map(mask_s2_clouds))

composite = s2.median().clip(aoi)
ndvi = composite.normalizedDifference(['B8','B4']).rename('NDVI')
ndmi = composite.normalizedDifference(['B8','B11']).rename('NDMI')
mndwi = composite.normalizedDifference(['B3','B11']).rename('MNDWI')
ndbi = composite.normalizedDifference(['B11','B8']).rename('NDBI')

Map.addLayer(composite, {'bands':['B4','B3','B2'], 'min':0, 'max':0.3}, 'Sentinel-2')
Map.addLayer(ndvi, {'min':-0.2, 'max':0.8}, 'NDVI')
Map

## 3. Sentinel-1 flood mapping starter

In [ ]:
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(aoi)
      .filter(ee.Filter.eq('instrumentMode','IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV'))
      .select('VV'))

pre = s1.filterDate('2025-05-01','2025-06-01').median().clip(aoi)
post = s1.filterDate('2025-06-01','2025-07-01').median().clip(aoi)
difference = post.subtract(pre).rename('VV_change')
candidate_flood = difference.lt(-3)

Map.addLayer(difference, {'min':-6,'max':6}, 'S1 VV change')
Map.addLayer(candidate_flood.selfMask(), {}, 'Candidate flood')
Map

## 4. Time series
Use geemap charts or export monthly NDVI statistics to pandas for trend analysis.